# NB07 — Interactive Azerbaijan Map & Dashboard

**Objective:** Build jury-ready interactive maps showing wildfire risk, weather forecasts, and city-level details for Azerbaijan.

### Features
- **Folium map** centred on Azerbaijan with one marker per city
- **Date selector** — generates one HTML map per forecast date
- **Colour coding** by wildfire risk level (green → yellow → orange → red)
- **Click popup** showing: city name, date, fire probability, risk level, temperature, humidity, wind, precipitation
- **Master dashboard** with dropdown for date and variable selection (Plotly)
- All maps saved as **standalone HTML** files for offline jury demo

**Input:** `outputs/wildfire_risk_30d.parquet`  
**Output:** `reports/maps/*.html`

In [ ]:
# ─── Cell 1: Imports ─────────────────────────────────────────────────────
import subprocess, sys, os, warnings, json
from pathlib import Path

for _p in ["folium", "plotly", "pandas", "numpy"]:
    try: __import__(_p)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _p])

import numpy as np
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.config import ROOT, OUTPUTS, CITIES, RISK_30D
MAPS = ROOT / "reports" / "maps"
MAPS.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")

In [ ]:
# ─── §1: Load risk data ──────────────────────────────────────────────────

risk = pd.read_parquet(RISK_30D)
risk["Date"] = pd.to_datetime(risk["Date"])
risk = risk.sort_values(["City", "Date"]).reset_index(drop=True)

# Ensure lat/lon
if "Latitude" not in risk.columns:
    coords = pd.DataFrame([{"City": c, "Latitude": lat, "Longitude": lon}
                            for c, (lat, lon) in CITIES.items()])
    risk = risk.merge(coords, on="City", how="left")

dates = sorted(risk["Date"].unique())
print(f"Risk data: {risk.shape}")
print(f"  Dates: {len(dates)} ({dates[0].date()} → {dates[-1].date()})")
print(f"  Cities: {risk['City'].nunique()}")
print(f"\nColumns: {list(risk.columns)}")

## §2 — Folium Maps (One Per Date)

Each date gets a standalone HTML map. Markers are colour-coded by risk level and sized by fire probability. Clicking a marker shows a detailed popup.

In [ ]:
# ─── §2: Generate Folium maps per date ───────────────────────────────────

RISK_COLORS = {
    "Low":     "green",
    "Medium":  "orange",
    "High":    "red",
    "Extreme": "darkred",
}

AZ_CENTER = [40.4, 47.8]
AZ_ZOOM   = 7


def make_folium_map(date_df, date_str):
    """Create a Folium map for one forecast date."""
    m = folium.Map(location=AZ_CENTER, zoom_start=AZ_ZOOM,
                   tiles="CartoDB positron")

    # Title
    title_html = f'''
    <div style="position: fixed; top: 10px; left: 60px; z-index: 1000;
                background: white; padding: 10px 20px; border-radius: 8px;
                box-shadow: 0 2px 6px rgba(0,0,0,0.3);
                font-family: Arial, sans-serif;">
        <h3 style="margin:0; color:#333;">Azerbaijan Wildfire Risk — {date_str}</h3>
        <p style="margin:2px 0 0 0; font-size:12px; color:#666;">
            Click a city for details | Colour = risk level | Size = probability
        </p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))

    for _, row in date_df.iterrows():
        city = row["City"]
        lat, lon = row["Latitude"], row["Longitude"]
        prob = row.get("fire_probability", 0)
        level = row.get("risk_level", "Low")
        color = RISK_COLORS.get(level, "gray")

        # Radius proportional to probability
        radius = max(8, prob * 40)

        # Build popup HTML
        popup_lines = [
            f"<b style='font-size:14px;'>{city}</b><br>",
            f"<b>Date:</b> {date_str}<br>",
            f"<b>🔥 Fire Probability:</b> {prob*100:.1f}%<br>",
            f"<b>Risk Level:</b> <span style='color:{color};font-weight:bold;'>{level}</span><br>",
            "<hr style='margin:4px 0;'>",
        ]

        # Weather data
        weather_cols = {
            "Temperature_C_mean": ("🌡 Temperature", "°C"),
            "Humidity_percent_mean": ("💧 Humidity", "%"),
            "Wind_Speed_kmh_mean": ("💨 Wind Speed", " km/h"),
            "Rain_mm_sum": ("🌧 Precipitation", " mm"),
            "Pressure_hPa_mean": ("📊 Pressure", " hPa"),
        }
        for col, (label, unit) in weather_cols.items():
            if col in row.index and pd.notna(row[col]):
                popup_lines.append(f"<b>{label}:</b> {row[col]:.1f}{unit}<br>")

        popup_html = "".join(popup_lines)

        folium.CircleMarker(
            location=[lat, lon],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{city}: {prob*100:.0f}% ({level})",
        ).add_to(m)

    # Legend
    legend_html = '''
    <div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
                background: white; padding: 10px 15px; border-radius: 8px;
                box-shadow: 0 2px 6px rgba(0,0,0,0.3); font-size: 12px;">
        <b>Risk Levels</b><br>
        <span style="color:green;">●</span> Low (&lt;15%)<br>
        <span style="color:orange;">●</span> Medium (15-35%)<br>
        <span style="color:red;">●</span> High (35-60%)<br>
        <span style="color:darkred;">●</span> Extreme (&gt;60%)
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    return m


# Generate maps for key dates (every 5th day + first and last)
key_dates = [dates[0]] + dates[4::5] + [dates[-1]]
key_dates = sorted(set(key_dates))

print(f"Generating {len(key_dates)} Folium maps...")

for dt in key_dates:
    date_str = pd.Timestamp(dt).strftime("%Y-%m-%d")
    day_data = risk[risk["Date"] == dt]

    if day_data.empty:
        continue

    fmap = make_folium_map(day_data, date_str)
    map_path = MAPS / f"fire_risk_{date_str}.html"
    fmap.save(str(map_path))
    print(f"  ✓ {date_str}: {len(day_data)} cities → {map_path.name}")

# Save "today" / first date as the main demo map
demo_map = make_folium_map(risk[risk["Date"] == dates[0]],
                           pd.Timestamp(dates[0]).strftime("%Y-%m-%d"))
demo_path = MAPS / "fire_risk_demo.html"
demo_map.save(str(demo_path))
print(f"\n  Demo map: {demo_path}")
print(f"  Total maps: {len(key_dates) + 1}")

## §3 — Plotly Interactive Dashboard (Date + Variable Selector)

A single HTML file with a dropdown for date and animated scatter-map showing all cities. This allows a jury member to explore any date interactively.

In [ ]:
# ─── §3: Plotly animated scatter-map dashboard ───────────────────────────

# Prepare data for Plotly
plot_df = risk.copy()
plot_df["date_str"] = plot_df["Date"].dt.strftime("%Y-%m-%d")
plot_df["fire_pct"] = (plot_df["fire_probability"] * 100).round(1)

# Assign numeric risk for colour scale
risk_num = {"Low": 0, "Medium": 1, "High": 2, "Extreme": 3}
plot_df["risk_num"] = plot_df["risk_level"].map(risk_num)

# Build hover text
hover_parts = []
for _, row in plot_df.iterrows():
    parts = [
        f"<b>{row['City']}</b>",
        f"Date: {row['date_str']}",
        f"Fire Risk: {row['fire_pct']}% ({row['risk_level']})",
    ]
    for col, label in [("Temperature_C_mean", "Temp"),
                        ("Humidity_percent_mean", "Humidity"),
                        ("Wind_Speed_kmh_mean", "Wind"),
                        ("Rain_mm_sum", "Rain")]:
        if col in row.index and pd.notna(row[col]):
            parts.append(f"{label}: {row[col]:.1f}")
    hover_parts.append("<br>".join(parts))

plot_df["hover"] = hover_parts

# Create animated scatter_geo
fig = px.scatter_geo(
    plot_df,
    lat="Latitude",
    lon="Longitude",
    color="fire_pct",
    size="fire_pct",
    hover_name="City",
    hover_data={"fire_pct": True, "risk_level": True,
                "Latitude": False, "Longitude": False, "date_str": False},
    animation_frame="date_str",
    color_continuous_scale=["green", "yellow", "orange", "red", "darkred"],
    range_color=[0, 80],
    size_max=30,
    title="Azerbaijan Wildfire Risk Forecast — 30 Day Outlook",
)

fig.update_geos(
    center=dict(lat=40.3, lon=47.8),
    projection_scale=15,
    showland=True, landcolor="rgb(243, 243, 243)",
    showocean=True, oceancolor="rgb(204, 229, 255)",
    showcountries=True, countrycolor="rgb(150, 150, 150)",
    showlakes=True, lakecolor="rgb(204, 229, 255)",
)

fig.update_layout(
    height=700,
    width=1000,
    coloraxis_colorbar=dict(title="Fire Risk %"),
    font=dict(family="Arial", size=12),
)

# Save as HTML
plotly_path = MAPS / "fire_risk_dashboard.html"
fig.write_html(str(plotly_path), include_plotlyjs="cdn")
print(f"Plotly dashboard saved: {plotly_path}")

fig.show()

## §4 — Weather Variable Maps

Additional maps showing temperature, humidity, and precipitation forecasts for each city. These let the jury understand *why* certain cities have higher risk.

In [ ]:
# ─── §4: Weather variable maps ───────────────────────────────────────────

WEATHER_VARS = {
    "Temperature_C_mean":      {"label": "Temperature (°C)", "cmap": "RdYlBu_r", "fmt": ".1f"},
    "Humidity_percent_mean":   {"label": "Humidity (%)",      "cmap": "YlGnBu",   "fmt": ".0f"},
    "Wind_Speed_kmh_mean":     {"label": "Wind Speed (km/h)", "cmap": "Purples",   "fmt": ".1f"},
    "Rain_mm_sum":             {"label": "Precipitation (mm)","cmap": "Blues",     "fmt": ".1f"},
}

available_vars = {k: v for k, v in WEATHER_VARS.items() if k in plot_df.columns}

# Create one animated map per variable
for var, cfg in available_vars.items():
    pdata = plot_df.dropna(subset=[var]).copy()
    if pdata.empty:
        continue

    fig_w = px.scatter_geo(
        pdata,
        lat="Latitude", lon="Longitude",
        color=var, size=np.abs(pdata[var]) + 1,
        hover_name="City",
        hover_data={var: True, "risk_level": True, "fire_pct": True,
                    "Latitude": False, "Longitude": False, "date_str": False},
        animation_frame="date_str",
        color_continuous_scale=cfg["cmap"],
        size_max=25,
        title=f"Azerbaijan {cfg['label']} Forecast — 30 Day",
    )

    fig_w.update_geos(
        center=dict(lat=40.3, lon=47.8),
        projection_scale=15,
        showland=True, landcolor="rgb(243, 243, 243)",
        showcountries=True, countrycolor="rgb(150, 150, 150)",
    )

    fig_w.update_layout(height=600, width=900,
                         font=dict(family="Arial", size=12))

    var_path = MAPS / f"weather_{var.split('_')[0].lower()}_dashboard.html"
    fig_w.write_html(str(var_path), include_plotlyjs="cdn")
    print(f"  ✓ {cfg['label']} → {var_path.name}")

print(f"\nAll maps saved to: {MAPS}")
print(f"\nFiles generated:")
for f in sorted(MAPS.glob("*.html")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")

print(f"\n→ Next: open 08_Climate_Fire_Analysis.ipynb")